# Logistic Blend: Monte Carlo + Ranking Points

Trains a logistic regression to combine MC win probability with ATP rolling ranking points.
Replaces the hardcoded `BLEND_W=0.5` linear blend with learned weights.

**Key design choice — mirroring:** The backtest stores the ATP winner as `is_player1=1`, so
`outcome=1` is true ~93% of the time. To get a balanced training set, every match is duplicated
with p1/p2 swapped (mc_prob → 1-mc_prob, log_ratio flipped, outcome flipped).

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler

In [ ]:
DB_PATH      = '../atp/data/tennis.db'
BACKTEST_CSV = 'brier_backtest_results.csv'

TRAIN_YEARS = [2023, 2024, 2025]
TEST_YEARS  = [2026]

## 1. Load backtest results and join tournament start dates

In [ ]:
bt = pd.read_csv(BACKTEST_CSV)
print(f'Backtest rows: {len(bt):,}')

conn = sqlite3.connect(DB_PATH)

dates_df = pd.read_sql_query("""
    SELECT m.id AS match_id, t.start_date
    FROM matches m
    JOIN tournaments t ON t.id = m.tournament_id
""", conn)

bt = bt.merge(dates_df, on='match_id', how='inner')
print(f'After joining start_date: {len(bt):,}')

## 2. Fetch rolling ranking points for each player as of tournament start date

In [ ]:
rankings = pd.read_sql_query("""
    SELECT player_id, rank_date, roll_points
    FROM player_rankings
    WHERE roll_points IS NOT NULL
    ORDER BY player_id, rank_date
""", conn)
conn.close()

rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])
bt['start_date']      = pd.to_datetime(bt['start_date'])

def get_points_as_of(player_id, date, rankings):
    """Most recent roll_points on or before date."""
    rows = rankings[(rankings.player_id == player_id) & (rankings.rank_date <= date)]
    if rows.empty:
        return np.nan
    return rows.iloc[-1]['roll_points']

# Build a per-(player, date) lookup to avoid repeated filtering
# Group rankings by player for fast slicing
rank_by_player = {pid: grp.set_index('rank_date')['roll_points']
                  for pid, grp in rankings.groupby('player_id')}

def lookup(player_id, date):
    s = rank_by_player.get(player_id)
    if s is None:
        return np.nan
    past = s[s.index <= date]
    return past.iloc[-1] if not past.empty else np.nan

bt['pts1'] = [lookup(row.p1_id, row.start_date) for _, row in bt.iterrows()]
bt['pts2'] = [lookup(row.p2_id, row.start_date) for _, row in bt.iterrows()]

missing = bt[['pts1','pts2']].isna().any(axis=1).sum()
print(f'Rows with missing ranking points: {missing} — dropping')
bt = bt.dropna(subset=['pts1','pts2'])
bt = bt[(bt.pts1 > 0) & (bt.pts2 > 0)]
print(f'Clean rows: {len(bt):,}')

## 3. Build features

In [ ]:
bt['log_ratio'] = np.log(bt['pts1'] / bt['pts2'])   # >0 when p1 is ranked higher
bt['rank_share'] = bt['pts1'] / (bt['pts1'] + bt['pts2'])  # for baseline comparison

print(bt[['p1_prob','log_ratio','rank_share','outcome']].describe())

## 4. Mirror dataset to fix outcome imbalance

The ATP scraper stores the winner as `is_player1`, so `outcome=1` ~93% of the time.
Mirroring creates a symmetric 50/50 dataset.

In [ ]:
original = bt.copy()
mirrored = bt.copy()
mirrored['p1_prob']    = 1 - bt['p1_prob']
mirrored['log_ratio']  = -bt['log_ratio']
mirrored['rank_share'] = 1 - bt['rank_share']
mirrored['outcome']    = 1 - bt['outcome']

full = pd.concat([original, mirrored], ignore_index=True)
print(f'Dataset size after mirroring: {len(full):,}')
print(f'Outcome balance: {full.outcome.mean():.3f} (should be 0.5)')

## 5. Train/test split by year

In [ ]:
train = full[full.event_year.isin(TRAIN_YEARS)]
test  = full[full.event_year.isin(TEST_YEARS)]

print(f'Train: {len(train):,} rows ({TRAIN_YEARS})')
print(f'Test : {len(test):,} rows ({TEST_YEARS})')

FEATURES = ['p1_prob', 'log_ratio']

X_train, y_train = train[FEATURES].values, train['outcome'].values
X_test,  y_test  = test[FEATURES].values,  test['outcome'].values

## 6. Fit logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression()
lr.fit(X_train_s, y_train)

print('Coefficients (scaled):')
for name, coef in zip(FEATURES, lr.coef_[0]):
    print(f'  {name:12s}: {coef:+.4f}')
print(f'  intercept   : {lr.intercept_[0]:+.4f}')

## 7. Evaluate: compare Brier scores on test set

Three baselines:
- **MC only**: raw `p1_prob` from Monte Carlo  
- **Ranking only**: `rank_share = pts1 / (pts1 + pts2)`  
- **Linear blend (current)**: `0.5 * mc + 0.5 * rank_share`  
- **Logistic regression**: learned combination of MC + log_ratio

In [ ]:
lr_probs = lr.predict_proba(X_test_s)[:, 1]

brier_mc      = brier_score_loss(y_test, test['p1_prob'].values)
brier_rank    = brier_score_loss(y_test, test['rank_share'].values)
brier_blend   = brier_score_loss(y_test, 0.5 * test['p1_prob'].values + 0.5 * test['rank_share'].values)
brier_lr      = brier_score_loss(y_test, lr_probs)

results = pd.DataFrame({
    'model':       ['MC only', 'Ranking only', 'Linear blend (0.5/0.5)', 'Logistic regression'],
    'brier_score': [brier_mc, brier_rank, brier_blend, brier_lr],
})
results['skill_score'] = 1 - results['brier_score'] / 0.25
print(results.to_string(index=False))

## 8. Per-year breakdown on test set

In [ ]:
test = test.copy()
test['lr_prob'] = lr_probs

for year, grp in test.groupby('event_year'):
    b_mc   = brier_score_loss(grp['outcome'], grp['p1_prob'])
    b_rank = brier_score_loss(grp['outcome'], grp['rank_share'])
    b_bl   = brier_score_loss(grp['outcome'], 0.5*grp['p1_prob'] + 0.5*grp['rank_share'])
    b_lr   = brier_score_loss(grp['outcome'], grp['lr_prob'])
    print(f'{year}  MC={b_mc:.4f}  Rank={b_rank:.4f}  Blend={b_bl:.4f}  LR={b_lr:.4f}  (n={len(grp):,})')